In [1]:
# Cell 1 — Imports and configuration

import pandas as pd
import numpy as np
import joblib

from pathlib import Path

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# Cell 2 — Load data and trained price model

DATA_PATH = Path("../data/processed/ml_ready_data.csv")
MODEL_PATH = Path("../models/ridge_price_model.pkl")
PREPROCESSOR_PATH = Path("../models/price_preprocessor.pkl")

df = pd.read_csv(DATA_PATH)

price_model = joblib.load(MODEL_PATH)
price_preprocessor = joblib.load(PREPROCESSOR_PATH)

print("========== MARKET RECOMMENDATION SETUP ==========")
print(f"Dataset rows       : {len(df):,}")
print(f"Dataset columns    : {df.shape[1]}")
print("Price model        : Loaded")
print("Preprocessor       : Loaded")

========== MARKET RECOMMENDATION SETUP ==========
Dataset rows       : 647,367
Dataset columns    : 20
Price model        : Loaded
Preprocessor       : Loaded


In [3]:
# Cell 3 — Create market candidates

market_candidates = (
    df.sort_values("Reported Date")
      .groupby(["Commodity", "Market Name"], as_index=False)
      .tail(1)
      .copy()
      .reset_index(drop=True)
)

print("========== MARKET CANDIDATES ==========")
print(f"Candidate rows: {len(market_candidates):,}")
print(f"Commodities: {market_candidates['Commodity'].nunique():,}")
print(f"Markets: {market_candidates['Market Name'].nunique():,}")

display(
    market_candidates[
        [
            "Commodity",
            "State Name",
            "District Name",
            "Market Name",
            "Modal Price (Rs./Quintal)",
            "Arrivals (Tonnes)"
        ]
    ].head(10)
)

========== MARKET CANDIDATES ==========
Candidate rows: 26,569
Commodities: 260
Markets: 2,621


,Commodity,State Name,District Name,Market Name,Modal Price (Rs./Quintal),Arrivals (Tonnes)
0,Red Gram,Maharashtra,Nagpur,Nagpur,1590.0,251.00
1,Chili Red,Tripura,Sepahijala,Bishalgarh,6000.0,0.04
2,Red Gram,Manipur,Imphal East,Lamlong Bazaar,3000.0,0.10
3,Onion,Bihar,Khagaria,Khagaria,1000.0,10.00
4,Green Peas,Uttar Pradesh,Sitapur,Sitapur,1450.0,1.50
5,Black Gram (Urd Beans)(Whole),Uttar Pradesh,Sitapur,Sitapur,1400.0,3.00
6,Bengal Gram(Gram)(Whole),Tripura,Sepahijala,Bishalgarh,2400.0,0.04
7,Green Peas,Jharkhand,West Singbhum,Chaibasa,1380.0,4.00
8,Barley (Jau),Uttar Pradesh,Rampur,Rampur,540.0,3.00
9,Brinjal,Madhya Pradesh,Dewas,Dewas,380.0,5.00


In [4]:
# Cell 4 — Predict next price for each market candidate

CANDIDATE_FEATURES = [
    "State Name",
    "District Name",
    "Market Name",
    "Variety",
    "Group",
    "Arrivals (Tonnes)",
    "Price_Lag_1",
    "Price_Lag_7",
    "Price_Rolling_7",
    "Price_Rolling_30",
    "Year",
    "Month",
    "Day",
    "DayOfWeek",
    "Price_Spread"
]

candidate_X = market_candidates[CANDIDATE_FEATURES].copy()

candidate_encoded = price_preprocessor.transform(candidate_X)

market_candidates["Predicted_Next_Price"] = price_model.predict(
    candidate_encoded
)

print("========== PRICE PREDICTIONS COMPLETE ==========")
print(f"Candidates scored: {len(market_candidates):,}")

display(
    market_candidates[
        [
            "Commodity",
            "State Name",
            "District Name",
            "Market Name",
            "Modal Price (Rs./Quintal)",
            "Predicted_Next_Price"
        ]
    ].head(10)
)

========== PRICE PREDICTIONS COMPLETE ==========
Candidates scored: 26,569


,Commodity,State Name,District Name,Market Name,Modal Price (Rs./Quintal),Predicted_Next_Price
0,Red Gram,Maharashtra,Nagpur,Nagpur,1590.0,1685.234422
1,Chili Red,Tripura,Sepahijala,Bishalgarh,6000.0,5868.071154
2,Red Gram,Manipur,Imphal East,Lamlong Bazaar,3000.0,3197.161088
3,Onion,Bihar,Khagaria,Khagaria,1000.0,1287.290445
4,Green Peas,Uttar Pradesh,Sitapur,Sitapur,1450.0,1874.221043
5,Black Gram (Urd Beans)(Whole),Uttar Pradesh,Sitapur,Sitapur,1400.0,1672.079344
6,Bengal Gram(Gram)(Whole),Tripura,Sepahijala,Bishalgarh,2400.0,2499.635913
7,Green Peas,Jharkhand,West Singbhum,Chaibasa,1380.0,1659.388686
8,Barley (Jau),Uttar Pradesh,Rampur,Rampur,540.0,1214.667379
9,Brinjal,Madhya Pradesh,Dewas,Dewas,380.0,956.118110


In [5]:
def calculate_net_realization(
    predicted_price_per_quintal,
    distance_km,
    quantity_kg,
    transport_rate_per_km,
    other_cost_per_kg=0
):
    """
    Calculate expected शुद्ध प्रापण मूल्य.

    predicted_price_per_quintal: predicted selling price
    distance_km: distance to market
    quantity_kg: farmer's quantity
    transport_rate_per_km: transport cost per km
    other_cost_per_kg: other selling costs per kg
    """

    predicted_price_per_kg = predicted_price_per_quintal / 100

    total_transport_cost = distance_km * transport_rate_per_km
    transport_cost_per_kg = total_transport_cost / quantity_kg

    net_price_per_kg = (
        predicted_price_per_kg
        - transport_cost_per_kg
        - other_cost_per_kg
    )

    total_net_realization = net_price_per_kg * quantity_kg

    return net_price_per_kg, total_net_realization


print("Net realization engine ready.")

Net realization engine ready.


In [6]:
# Farmer scenario
FARMER_CROP = "Tomato"
FARMER_QUANTITY_KG = 500

# Example transport assumption for prototype
TRANSPORT_RATE_PER_KM = 25
OTHER_COST_PER_KG = 0.50

tomato_candidates = market_candidates[
    market_candidates["Commodity"].str.strip().str.lower() == FARMER_CROP.lower()
].copy()

print("========== FARMER SCENARIO ==========")
print(f"Crop              : {FARMER_CROP}")
print(f"Quantity          : {FARMER_QUANTITY_KG} kg")
print(f"Tomato markets    : {len(tomato_candidates):,}")

display(
    tomato_candidates[
        [
            "Commodity",
            "State Name",
            "District Name",
            "Market Name",
            "Predicted_Next_Price"
        ]
    ].head(10)
)

========== FARMER SCENARIO ==========
Crop              : Tomato
Quantity          : 500 kg
Tomato markets    : 970


,Commodity,State Name,District Name,Market Name,Predicted_Next_Price
124,Tomato,West Bengal,Burdwan,Katwa,2395.908267
149,Tomato,Maharashtra,Chandrapur,Chandrapur,1006.552686
154,Tomato,Gujarat,Vadodara(Baroda),Vadodara,1027.452153
190,Tomato,Gujarat,Dahod,Dahod,2048.268672
247,Tomato,Gujarat,Mehsana,Mehsana,1099.795312
277,Tomato,Gujarat,Bharuch,Amod,1284.011510
308,Tomato,Andhra Pradesh,East Godavari,Rajahmundry,2162.236452
376,Tomato,Andhra Pradesh,Nellore,Nellore,1966.369518
459,Tomato,Madhya Pradesh,Chhindwara,Chhindwara,796.688171
483,Tomato,Jammu and Kashmir,Anantnag,Vessue,1302.309485


In [8]:
# Prototype distance assumptions (for demonstration only)
# These are NOT live GPS distances.

distance_map = {
    "Katwa": 120,
    "Chandrapur": 80,
    "Vadodara": 150,
    "Dahod": 100,
    "Mehsana": 180,
    "Amod": 140,
    "Rajahmundry": 200,
    "Nellore": 160,
    "Chhindwara": 90,
    "Vessue": 130
}

tomato_candidates["Distance_km"] = (
    tomato_candidates["Market Name"]
    .map(distance_map)
)

# Keep only markets with a demo distance
recommendations = tomato_candidates.dropna(
    subset=["Distance_km"]
).copy()

# Calculate expected net realization
results = recommendations.apply(
    lambda row: calculate_net_realization(
        predicted_price_per_quintal=row["Predicted_Next_Price"],
        distance_km=row["Distance_km"],
        quantity_kg=FARMER_QUANTITY_KG,
        transport_rate_per_km=TRANSPORT_RATE_PER_KM,
        other_cost_per_kg=OTHER_COST_PER_KG
    ),
    axis=1
)

recommendations["Net_Price_per_kg"] = results.apply(lambda x: x[0])
recommendations["Total_Net_Realization"] = results.apply(lambda x: x[1])

recommendations = recommendations.sort_values(
    "Net_Price_per_kg",
    ascending=False
).reset_index(drop=True)

print("========== MARKET RECOMMENDATIONS ==========")
print(f"Markets evaluated: {len(recommendations)}")

display(
    recommendations[
        [
            "Commodity",
            "State Name",
            "District Name",
            "Market Name",
            "Predicted_Next_Price",
            "Distance_km",
            "Net_Price_per_kg",
            "Total_Net_Realization"
        ]
    ].head(10)
)

========== MARKET RECOMMENDATIONS ==========
Markets evaluated: 10


,Commodity,State Name,District Name,Market Name,Predicted_Next_Price,Distance_km,Net_Price_per_kg,Total_Net_Realization
0,Tomato,West Bengal,Burdwan,Katwa,2395.908267,120.0,17.459083,8729.541333
1,Tomato,Gujarat,Dahod,Dahod,2048.268672,100.0,14.982687,7491.343358
2,Tomato,Andhra Pradesh,Nellore,Nellore,1966.369518,160.0,11.163695,5581.847589
3,Tomato,Andhra Pradesh,East Godavari,Rajahmundry,2162.236452,200.0,11.122365,5561.182258
4,Tomato,Jammu and Kashmir,Anantnag,Vessue,1302.309485,130.0,6.023095,3011.547424
5,Tomato,Maharashtra,Chandrapur,Chandrapur,1006.552686,80.0,5.565527,2782.763431
6,Tomato,Gujarat,Bharuch,Amod,1284.011510,140.0,5.340115,2670.057549
7,Tomato,Madhya Pradesh,Chhindwara,Chhindwara,796.688171,90.0,2.966882,1483.440855
8,Tomato,Gujarat,Vadodara(Baroda),Vadodara,1027.452153,150.0,2.274522,1137.260764
9,Tomato,Gujarat,Mehsana,Mehsana,1099.795312,180.0,1.497953,748.976561


In [9]:
# Select the best evaluated market
best_market = recommendations.iloc[0]

print("========== MANDISAARTHI RECOMMENDATION ==========")
print()
print(f"Crop              : {FARMER_CROP}")
print(f"Quantity          : {FARMER_QUANTITY_KG} kg")
print()
print(f"Recommended Market: {best_market['Market Name']}")
print(f"State             : {best_market['State Name']}")
print(f"District          : {best_market['District Name']}")
print(f"Predicted Price   : ₹{best_market['Predicted_Next_Price']:,.2f}/quintal")
print(f"Estimated Distance: {best_market['Distance_km']:.0f} km")
print(f"Net Price         : ₹{best_market['Net_Price_per_kg']:.2f}/kg")
print(f"Total Net Value   : ₹{best_market['Total_Net_Realization']:,.2f}")
print()
print(
    f"Based on the evaluated candidates, {best_market['Market Name']} "
    f"has the highest estimated शुद्ध प्रापण मूल्य for this scenario."
)

========== MANDISAARTHI RECOMMENDATION ==========

Crop              : Tomato
Quantity          : 500 kg

Recommended Market: Katwa
State             : West Bengal
District          : Burdwan
Predicted Price   : ₹2,395.91/quintal
Estimated Distance: 120 km
Net Price         : ₹17.46/kg
Total Net Value   : ₹8,729.54

Based on the evaluated candidates, Katwa has the highest estimated शुद्ध प्रापण मूल्य for this scenario.


In [10]:
# Final farmer-facing recommendation

best_market = recommendations.iloc[0]

print("========== MANDISAARTHI RECOMMENDATION ==========")
print()
print(f"🌾 Crop              : {FARMER_CROP}")
print(f"📦 Quantity          : {FARMER_QUANTITY_KG} kg")
print()
print(f"📍 Recommended Market: {best_market['Market Name']}")
print(f"🏛️ State             : {best_market['State Name']}")
print(f"📌 District          : {best_market['District Name']}")
print(
    f"💰 Predicted Price   : "
    f"₹{best_market['Predicted_Next_Price']:,.2f}/quintal"
)
print(
    f"🚚 Estimated Distance: "
    f"{best_market['Distance_km']:.0f} km"
)
print(
    f"💵 शुद्ध प्रापण मूल्य: "
    f"₹{best_market['Net_Price_per_kg']:.2f}/kg"
)
print(
    f"💰 Estimated Total   : "
    f"₹{best_market['Total_Net_Realization']:,.2f}"
)
print()
print(
    "Recommendation is based on the evaluated market candidates "
    "and prototype logistics assumptions."
)

========== MANDISAARTHI RECOMMENDATION ==========

🌾 Crop              : Tomato
📦 Quantity          : 500 kg

📍 Recommended Market: Katwa
🏛️ State             : West Bengal
📌 District          : Burdwan
💰 Predicted Price   : ₹2,395.91/quintal
🚚 Estimated Distance: 120 km
💵 शुद्ध प्रापण मूल्य: ₹17.46/kg
💰 Estimated Total   : ₹8,729.54

Recommendation is based on the evaluated market candidates and prototype logistics assumptions.


In [11]:
# Final validation of the recommendation pipeline

print("========== FINAL VALIDATION ==========")

checks = {
    "Price predictions generated": (
        "Predicted_Next_Price" in recommendations.columns
        and recommendations["Predicted_Next_Price"].notna().all()
    ),
    "Net price calculated": (
        "Net_Price_per_kg" in recommendations.columns
        and recommendations["Net_Price_per_kg"].notna().all()
    ),
    "Total net realization calculated": (
        "Total_Net_Realization" in recommendations.columns
        and recommendations["Total_Net_Realization"].notna().all()
    ),
    "Recommendations sorted": (
        recommendations["Net_Price_per_kg"].is_monotonic_decreasing
    ),
}

for check, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {check}")

print()
print(f"Markets evaluated : {len(recommendations):,}")
print(f"Best market       : {recommendations.iloc[0]['Market Name']}")
print(
    f"Best net price    : "
    f"₹{recommendations.iloc[0]['Net_Price_per_kg']:.2f}/kg"
)

print()
print("========== NOTEBOOK 04 COMPLETE ==========")

========== FINAL VALIDATION ==========
PASS: Price predictions generated
PASS: Net price calculated
PASS: Total net realization calculated
PASS: Recommendations sorted

Markets evaluated : 10
Best market       : Katwa
Best net price    : ₹17.46/kg

========== NOTEBOOK 04 COMPLETE ==========
